# Regression Modeling

## Overview

In this section we're going to try various approaches to try and prove our whether or not we can predict a neighborhoods noise level based on various socioeconomic factors.

$H_0$: There is no association between a ZCTA's socioeconomic disadvantage and its average transportation noise exposure ($\beta_1 = 0$), controlling for geographic and demographic factors.

$H_1$: ZCTAs with higher levels of socioeconomic disadvantage (lower median household income, higher poverty rates, or higher disadvantage index scores) experience significantly higher mean transportation noise levels ($\beta_1 > 0$ or $\beta_1 \neq 0$).

### Pre-modeling observations

The most important pre-modeling observation is that decibels (dB) which we're trying to predict is not a linear variable, it actually grows exponentially like the earthquake magnitude scale. 30 decibels is 10 times louder than 20 decibles. 

Another important pre-modeling observation that we already established in the EDA section is that there is some pretty strong correlation between similar variables like income, poverty_rate, etc. 

Imports:

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import acf
import numpy as np

## Original simple multi-linear regression model


In [27]:
master = pd.read_csv("../data/processed/master_dataset.csv")
master.head()

,zcta,noise_mean_db,noise_max_db,noise_min_db,median_household_income,unemployment_rate,population,pnhwhite,pnhblack,phispanic,pforeign_born,punemployed,affluence,disadvantage,median_family_income,home_value,poverty_rate
0,90001,55.280064,74.542302,45.102884,60767.0,54.3,57652.0,0.006470,0.069347,0.913099,0.409283,0.100026,0.111688,0.246609,59176.0,5.938812e+05,16.7
1,90002,55.551904,74.367805,45.061863,59021.0,55.3,53108.0,0.003766,0.152444,0.824094,0.347273,0.112967,0.107083,0.298019,57212.0,5.949507e+05,21.4
2,90003,57.612337,83.350455,45.052543,56030.0,55.6,75024.0,0.004718,0.161562,0.815459,0.398219,0.095017,0.100366,0.325037,53272.0,6.071944e+05,23.3
3,90004,56.249286,74.941374,45.100273,64826.0,64.3,58833.0,0.205803,0.041966,0.465895,0.465130,0.073530,0.334647,0.226329,63444.0,1.450675e+06,13.9
4,90005,54.854745,69.565419,45.163183,49419.0,65.3,37754.0,0.101155,0.049743,0.492742,0.540976,0.047240,0.311810,0.237600,58545.0,8.944448e+05,17.3


In [28]:
#noise_max and noise_min encode too similar info to noise_mean so we're taking it out
predictors = [col for col in master.columns if 
              col != 'noise_mean_db' and col != 'noise_max_db' and col != 'noise_min_db'] 

formula = f"noise_mean_db ~ {' + '.join(predictors)}"
original_model = smf.ols(formula, data=master).fit()
print(original_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     11.89
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           1.85e-26
Time:                        11:57:24   Log-Likelihood:                -2688.1
No. Observations:                1407   AIC:                             5406.
Df Residuals:                    1392   BIC:                             5485.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

#### Original Model Analysis:

This model is actually not half bad! Our $R^2$ value is pretty low, but it's the difference in $R^2$ and adj. $R^2$ that we really care about. The fact that there's not a big difference after adjusting means we have some good variables

The prob F-Stat is amazing, there's definitely some statistically significant analysis here. 

Some of our coefficients though are kinda weird. Likely because of the strong correlation between various variables and also because our sample size is not the biggest.

Lastly, we get a warning about our condition number being huge, this is telling us again that there exists strong multicollinearity among some of our variables. 

### We're going to explore two options for bettering our basic log-linear model

    1. Choosing columns manually

    2. Choosing columns using recommendations from Lasso regression

The obvious place to start is just using our pre existing knowledge of the data to make a decision of what variables to keep. Variables that sound redundant, likely are. 

## Section A: Manually Dropping Rows

In [29]:
columnds = master.columns
columnds

Index(['zcta', 'noise_mean_db', 'noise_max_db', 'noise_min_db',
       'median_household_income', 'unemployment_rate', 'population',
       'pnhwhite', 'pnhblack', 'phispanic', 'pforeign_born', 'punemployed',
       'affluence', 'disadvantage', 'median_family_income', 'home_value',
       'poverty_rate'],
      dtype='object')

There's 3 big categories to our data:

1. Socio-economic stattus: median_household_income, affluence, median_household_income, home_value
2. Poverty: unemployment rate, punemployed, disadvantage, poverty_rate
3. Race: pnhwhite, pnhblack, phispanic, pforeigh_born

And then there's our other singular features like:

4. zcta
5. population

As a reminder, our current analysis focuses just on mean noise so we're ignoring max and min.

So what makes most sense here:

For socio-economic status we could choose either median_household_income or median_household_income income, let's go with household! home_value is probably useful enough to keep too!

For poverty, poverty_rate is the obvious choice. It encapsulates unemployment rate and is less abstract than disadvantage.

For race it's probably fine to just drop pnhwhite and keep the rest. pnhwhite is just redundant info.

We should also drop zcta since the numerical number doesn't mean anything, but we should definitely keep population.


So our fun new model looks like this:

In [30]:
predictors = ['median_household_income', 'home_value', 'poverty_rate', 'pnhblack', 
              'phispanic', 'pforeign_born', 'population']

formula = f"noise_mean_db ~ {' + '.join(predictors)}"
improved_model = smf.ols(formula, data=master).fit()
print(improved_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.076
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     16.45
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           6.02e-21
Time:                        11:57:24   Log-Likelihood:                -2711.9
No. Observations:                1407   AIC:                             5440.
Df Residuals:                    1399   BIC:                             5482.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

#### Manual Model Analysis:

This model is a bit better in some ways! 

Our condition number is smaller but still too big. but 

Our $R^2$ went down quite a bit though.

Our F-Stat got better but not by a lot.

I think we can do a lot better actually using techniques we learned in class. 

## Section B: Lasso Regression


In [31]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

predictors = [col for col in master.columns if col not in ['noise_mean_db', 'noise_max_db', 'noise_min_db']]
all_needed_columns = ['noise_mean_db'] + predictors

#drop rows with NaN values
master_clean = master[all_needed_columns].dropna().copy()

y = master_clean['noise_mean_db']

#standardize variables
df_scaled = master_clean[predictors].copy()
for col in predictors:
    if col != 'zcta':
        df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()

df_scaled['noise_mean_db'] = y

formula = f"noise_mean_db ~ {' + '.join(predictors)}"
model = smf.ols(formula, data=df_scaled)

alphas = np.linspace(0.001, 1.0, 100)

best_alpha = None
lowest_bic = float('inf')  
best_params = None

y_true = df_scaled['noise_mean_db']
X_matrix = model.exog 
n_samples = len(y_true)

#using BIC to find the best alpha
for a in alphas:
    test_lasso = model.fit_regularized(alpha=a, L1_wt=1.0)
    
    y_pred = np.dot(X_matrix, test_lasso.params)
    sse = np.sum((y_true - y_pred) ** 2)
    
    k_features = np.sum(test_lasso.params[1:] != 0)
    
    if sse > 0:
        bic = n_samples * np.log(sse / n_samples) + k_features * np.log(n_samples)
    else:
        bic = float('inf')
    
    #keep best alpha
    if bic < lowest_bic:
        lowest_bic = bic
        best_alpha = a
        best_params = test_lasso.params

print(f"The optimal alpha value is: {best_alpha}")

final_coefs = pd.DataFrame({
    'Feature': ['Intercept'] + predictors,
    'Coefficient': best_params
})

print([final_coefs['Coefficient']])

The optimal alpha value is: 0.05145454545454546
[Intercept                  53.204694
zcta                        0.000025
median_household_income     0.000000
unemployment_rate           0.180408
population                  0.000000
pnhwhite                   -0.168996
pnhblack                    0.100716
phispanic                   0.000000
pforeign_born               0.165322
punemployed                 0.000000
affluence                   0.127686
disadvantage                0.000000
median_family_income        0.000000
home_value                  0.000000
poverty_rate                0.000000
Name: Coefficient, dtype: float64]


Yippee!! We've figured out which variables are worth keeping, we can create a new, hopefully improved linear regression model based on just the coefficients that didn't go to 0.


In [32]:
features = ['zcta', 'unemployment_rate', 'pnhwhite', 
            'pnhblack', 'pforeign_born', 'affluence']
formula = f"noise_mean_db ~ {' + '.join(features)}"
lasso_improved_model = smf.ols(formula, data=master).fit()
print(lasso_improved_model.summary())

                            OLS Regression Results                            
Dep. Variable:          noise_mean_db   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     30.13
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           2.83e-34
Time:                        11:57:25   Log-Likelihood:                -2907.2
No. Observations:                1512   AIC:                             5828.
Df Residuals:                    1505   BIC:                             5866.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            65.2262      2.53

#### Lasso Model Analysis:

We really improved our model! I mean, is it a great model? Not really... but we definitely did a lot better than just manually selecting and have addressed some of the issues with our original model.

Our condition number got way smaller! It's still a bit too big and there is still likely multicollinearity in our predictors but at least that plays way less of a factor.

Our $R^2$ stayed the same and our adjusted $R^2$ actually went up! We literally improved the predictive power of our model.

And our F-Stat is huge which is a great sign for statistical significance of the model.

### Conclusion:

At the end of the day, our model was hampered by one big issue and that is data! More data is always a good thing, unless you don't have the space to store it or the memory to do analysis on it. Nevertheless, I think we did a pretty good job!

There is a clear correlation between noise leveles and various socio-economic factors! Richer neighborhoods are quiter while neighborhoods with high unemployment rates or high percentages of foreign born residents tend to be louder. 

We saw clear model improvements after trying Lasso regression, but we also saw that even just having baseline knowledge about the data you're working with and trying to manually select can make a difference too!

If we would do this project all over again, we'd start with making our sample size bigger, maybe expanding to neighborhoods across the US instead of only focusing on California. If we were really serious data scientists we may even try some solutions to filling our NaN rows with reasonable inferences using some ML algorithm. 